In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import torch
# we perform Tucker Decomposition using tensorly
import tensorly as tl
import tensorly_custom as tlc
import time, os, pickle
from utils import DATA_DIR, notify_discord, select_gpu, torch_or_pickle_load, tree_to_device

In [3]:
device = select_gpu()
tl.set_backend('pytorch')

# we load in the different source tensors
tensor_path = f"{DATA_DIR}/tensors/fineweb_sparse/populated/sii_1250.pt"
tensor = torch_or_pickle_load(tensor_path)
# with open(f"{path_to_tensors}/tensor_{population_method}_{top_k}.pkl", "rb") as f:
#     # these were saved as torch tensors!
#     try:
#         tensor = pickle.load(f)
#         # we make sure the tensor is a torch tensor
#         if not isinstance(tensor, torch.Tensor):
#             tensor = torch.as_tensor(tensor)
#     except pickle.UnpicklingError as e:
#         # print("Pickle unpickling error, trying torch.load instead.")
#         tensor = torch.load(f)
print(f"Loaded tensor of shape {tensor.shape}")
init = "svd"
print(f"Using {init} initialization for decomposition.")

Selected GPU 1 with 20189.94 MB used memory.
Loaded tensor of shape torch.Size([1250, 1250, 1250])
Using svd initialization for decomposition.


In [4]:
tensor = tree_to_device(tensor, "cuda:0")

In [5]:
# we check if we have a sparse tensor that is on cuda
tensor.is_cuda, tensor.is_sparse

(True, True)

In [6]:
rank = [150, 150, 150]
n_iter_max = 5
tol = 1e-5
random_state = None

# Recreating non_negative tucker step by step

In [7]:
# we set the backend to sparse
import tensorly_custom as tlc
tlc.set_backend('numpy')

In [8]:
import tensorly_custom.decomposition._tucker as tlc_decomp

# rank = validate_tucker_rank(tl.shape(tensor), rank=rank) # we skip this validation for now
assert rank == list(rank), "Rank should be a list of integers."
assert len(rank) == tlc.ndim(
    tensor
), "Length of rank should be equal to the number of dimensions of the tensor."

epsilon = 10e-12

# Initialisation
nn_core, nn_factors = tlc_decomp.initialize_tucker(
    tensor,
    rank,
    range(tlc.ndim(tensor)),
    init="svd",
    random_state=random_state,
    non_negative=True,
)


TypeError: transpose() received an invalid combination of arguments - got (list), but expected one of:
 * (int dim0, int dim1)
 * (name dim0, name dim1)


In [9]:
# we transform our pytorch sparse tensor into a numpy sparse tensor
import numpy as np
import sparse

indices = tensor._indices().cpu().numpy()
print(indices.shape)
values = tensor._values().cpu().numpy()
print(values.shape)
shape = tensor.shape
np_sparse = sparse.COO(indices, values, shape=shape)
print(np_sparse)

(3, 254185)
(254185,)
<COO: shape=(1250, 1250, 1250), dtype=float32, nnz=254185, fill_value=0.0>


In [10]:
from tensorly_custom.contrib.sparse import tensor as sparse_tensor
np_sparse = sparse_tensor(np_sparse)
np_sparse

Format,coo
Data Type,float32
Shape,"(1250, 1250, 1250)"
nnz,254185
Density,0.00013014272
Read-only,True
Size,6.8M
Storage ratio,0.00


In [1]:
from tensorly_custom.contrib.sparse.decomposition import non_negative_tucker as sparse_non_negative_tucker
tlc.set_backend("numpy")
# sparse_non_negative_tucker(
#     np_sparse,
#     rank,
#     n_iter_max=10,
#     init="random",
#     tol=10e-5,
#     random_state=None,
#     verbose=False,
#     return_errors=False,
#     normalize_factors=False,)
# we print the content of sparse_non_negative_tucker
import inspect
print(inspect.getsource(sparse_non_negative_tucker.__wrapped__))
print(inspect.getsource(sparse_non_negative_tucker))

NameError: name 'tlc' is not defined

In [ ]:
# what we want to do with sparse tensors:
def non_negative_tucker(
    tensor,
    rank,
    n_iter_max=10,
    init="svd",
    tol=10e-5,
    random_state=None,
    verbose=False,
    return_errors=False,
    normalize_factors=False,
):

    norm_tensor = tl.norm(tensor, 2)
    rec_errors = []

    for iteration in range(n_iter_max):
        for mode in range(tl.ndim(tensor)):
            B = tucker_to_tensor((nn_core, nn_factors), skip_factor=mode)
            B = tl.transpose(unfold(B, mode))

            numerator = tl.dot(unfold(tensor, mode), B)
            numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
            denominator = tl.dot(nn_factors[mode], tl.dot(tl.transpose(B), B))
            denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
            nn_factors[mode] *= numerator / denominator

        numerator = tucker_to_tensor((tensor, nn_factors), transpose_factors=True)
        numerator = tl.clip(numerator, a_min=epsilon, a_max=None)
        for i, f in enumerate(nn_factors):
            if i:
                denominator = mode_dot(denominator, tl.dot(tl.transpose(f), f), i)
            else:
                denominator = mode_dot(nn_core, tl.dot(tl.transpose(f), f), i)
        denominator = tl.clip(denominator, a_min=epsilon, a_max=None)
        nn_core *= numerator / denominator

        rec_error = (
            tl.norm(tensor - tucker_to_tensor((nn_core, nn_factors)), 2) / norm_tensor
        )
        rec_errors.append(rec_error)
        if iteration > 1 and verbose:
            print(
                f"reconstruction error={rec_errors[-1]}, variation={rec_errors[-2] - rec_errors[-1]}."
            )

        if iteration > 1 and tl.abs(rec_errors[-2] - rec_errors[-1]) < tol:
            if verbose:
                print(f"converged in {iteration} iterations.")
            break
        if normalize_factors:
            nn_core, nn_factors = tucker_normalize((nn_core, nn_factors))
    tensor = TuckerTensor((nn_core, nn_factors))
    if return_errors:
        return tensor, rec_errors
    else:
        return tensor


In [3]:


with live_iteration_numbers():
    (core, factors), errors_reg = non_negative_tucker(
        tensor, non_negative=True, hals=False,
        rank=rank, n_iter_max=n_iter_max, init=init, tol=tol, verbose=verbose, return_errors=True
    )
# we move the results to cpu
(core, factors) = tree_to_device((core, factors), torch.device("cpu"))
# we save this to disk using torch
torch.save((core, factors), f"{decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pt")
print(f"Saved decomposition to {decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pt")
# with open(f"{decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pkl", "wb") as f:
#     pickle.dump((core, factors), f)
# print(f"Saved decomposition to {decomposition_path}/{population_method}_{top_k}d_{rank}r_{n_iter_max}i.pkl")
print(f"Final reconstruction error: {errors_reg[-1]}")


if __name__ == "__main__":
paths = [DATA_DIR/"tensors/fineweb_dutch_vectors_ids/sparse"]
# ranks = [100, 150]
# top_ks = [750, 1000, 1250]
# population_methods = ["counting", "sc", "sii"]

ranks = [100]
top_ks = [1250]
population_methods = ["sc", "sii"]
start_time = time.time()
for path in paths:
    for top_k in top_ks:
        for population_method in population_methods:
            for rank in ranks:
                try:
                    start_element = time.time()
                    init = "random" if top_k >= 999 else "svd"
                    print(f"\n\nStarting {population_method} tensor decomposition with rank={rank} for top_k={top_k}...")
                    (_, _), errors = tensor_iteration(
                        path_to_tensors=path,
                        population_method=population_method,
                        top_k=top_k,
                        n_iter_max=1000,
                        rank=rank,
                        tol=1e-10,
                        verbose=True,
                        init=init,
                        gpu_device=None,
                        override='/home/local/stefa/data/tensors/fineweb_dutch_vectors_ids/sparse/count_tensor.pt'
                    )
                    # we save the errors
                    decomposition_path = f"{path}/decomposition"
                    os.makedirs(decomposition_path, exist_ok=True)
                    with open(f"{decomposition_path}/errors_{population_method}_{top_k}d_{rank}r_1000i.pkl", "wb") as f:
                        pickle.dump(errors, f)
                    print(f"Saved errors to {decomposition_path}/errors_{population_method}_{top_k}d_{rank}r_1000i.pkl")

                    # we delete cuda cache to avoid OOM issues
                    torch.cuda.empty_cache()
                    end_element = time.time()
                    notify_discord(f"✅ Completed tensor decomposition for {population_method} with top_k={top_k}, "
                                   f"rank={rank} in {(end_element - start_element)} seconds.")
                except Exception as e:
                    notify_discord(f"❌ Error during tensor decomposition for {population_method} with top_k={top_k}, rank={rank}: {e}")
end_time = time.time()
notify_discord(f"✅ Tensor decompositions completed for all specified configurations"
               f" in {(end_time-start_time)} seconds."
               )

IndentationError: expected an indented block after 'if' statement on line 137 (1654665937.py, line 138)